# 04 — Leakage, categorical cardinality & AIRPORT_INFO (team 07)

**Owner:** Xiaojie · **Branch:** `team-07-xiaojie`

**Questions to answer**
1. **Leakage audit** — for every column: would we know it *before* the month we're forecasting?
   Anything computed from the actual passenger count (or recorded after it) is a leak. List each
   suspect column and the reason. (Remember D2: a leaking feature can score 0.999 and be worthless.)
2. **Categorical cardinality** — how many unique values does each text column have? (Airports,
   routes, carriers…) High cardinality changes encoding: one-hot vs target/frequency encoding vs trees.
3. **AIRPORT_INFO** — what's in it, what key joins it to `pax` (origin and destination airport?),
   and do all airports in `pax` have a match? Join it and check that the row count doesn't change.
4. Which external sources from the PM's list (census population, FAA enplanements…) would add
   something the data doesn't already have? (Just a recommendation — don't download anything new
   without checking with Patrick first; new sources need a DATA.md entry.)

**Write the "why", not just the plot.** Every plot or number needs a sentence after it:
*"…therefore we should ___ when we model."* Those sentences go in the table at the bottom.

> **NDA reminder.** This is real American Airlines data. Never commit it, never paste rows or
> screenshots of it anywhere outside the team, and **clear outputs before every commit**
> (Kernel → Restart & Clear Output) so no data ends up in the notebook file.

In [ ]:
%run ./00_load_and_preprocess.ipynb
# gives you: pax, airports, TARGET, REPO_ROOT, RESULTS_DIR

## 1. Leakage audit

In [ ]:
reviews = {
    "Unnamed: 0": ("N/A", "Saved row index. Exclude from model features."),
    "DEP_MONTH": ("Yes", "The forecast month is known in advance."),
    "DEP_YEAR": ("Yes", "The forecast year is known in advance."),
    "ORG_AIRPRT": (
        "Yes, if route is set in advance",
        "The origin airport is known once the forecast route is selected.",
    ),
    "DES_AIRPRT": (
        "Yes, if route is set in advance",
        "The destination airport is known once the forecast route is selected.",
    ),
    TARGET: (
        "No",
        "This is the actual passenger count for the forecast month. Use it as y.",
    ),
    "OD_REV_USD": (
        "No",
        "It is unavailable before that month. Exclude from model features.",
    ),
    "SF_LOAD_TMS": (
        "N/A",
        "Snowflake load timestamp. Exclude from model features.",
    ),
}

rows = []
for column in pax.columns:
    known, reason = reviews.get(column, ("Confirm", "Check definition and timing."))
    rows.append({
        "Column": column,
        "Known before forecast month?": known,
        "Reason": reason,
    })

leakage_audit = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", None):
    display(leakage_audit)

Departure year and month are known in advance, so keep both as calendar features. Keep origin and destination airport codes if the routes are selected before the forecast month. Use `OD_PAX` as the target (`y`). Exclude `OD_REV_USD` from the feature set (`X`) because it contains actual revenue for the same route and month being forecast. Also exclude `Unnamed: 0` and `SF_LOAD_TMS` because they are a saved row index and a constant load timestamp.

## 2. Cardinality

In [ ]:
text_cols = pax.select_dtypes(include=["object", "string", "category"]).columns

# Count distinct values in each text column.
cardinality = pax[text_cols].nunique().sort_values(ascending=False)
display(cardinality.to_frame("Unique values"))

# Show the five most frequent values in each text column.
for col in text_cols:
    print(f"\n{col}")
    display(pax[col].value_counts().head(5).to_frame("Row count"))

route_count = pax[["ORG_AIRPRT", "DES_AIRPRT"]].dropna().drop_duplicates().shape[0]
print("Distinct routes:", route_count)

The passenger data has 150 distinct airport codes in the origin column and 150 in the destination column. The same airport can appear in both roles, so one-hot encoding the columns separately gives each airport a distinct indicator for each role it appears in. One-hot encoding the 17119 distinct origin–destination pairs would create about 57 times as many columns. I’d start by encoding origin and destination separately to keep the feature set smaller and let the model learn from airports that appear across multiple routes.

## 3. AIRPORT_INFO join

In [ ]:
display(airports.head())
print(airports.columns.tolist())

airport_key = "AIRPRT_CD"

# An airport lookup row without a code cannot identify an airport.
airport_lookup = airports.dropna(subset=[airport_key])
print("Duplicate airport codes in lookup:", int(airport_lookup[airport_key].duplicated().sum()))
merged = pax

for side, pax_key in [("origin", "ORG_AIRPRT"), ("dest", "DES_AIRPRT")]:
    lookup = airport_lookup.add_prefix(f"{side}_")
    match_col = f"{side}_match"

    merged = merged.merge(
        lookup,
        how="left",
        left_on=pax_key,
        right_on=f"{side}_{airport_key}",
        validate="many_to_one",
        indicator=match_col,
    )

    assert len(merged) == len(pax), f"{side} join changed the row count"

    unmatched = merged[match_col].eq("left_only")
    print(f"{side}: {unmatched.sum():,} unmatched rows ({unmatched.mean():.1%})")

print("Same row count:", len(merged) == len(pax))

# Missing values in the airport attributes the join added (check before adding them to X)
added_cols = [col for col in merged.columns if col not in pax.columns and not col.endswith("_match")]
display(merged[added_cols].isna().mean().sort_values(ascending=False).to_frame("Share missing"))

I joined `AIRPORT_INFO` to `pax` twice using `AIRPRT_CD`, once for the origin and once for the destination. The lookup had no duplicate airport codes, every origin and destination airport in `pax` matched an entry in `AIRPORT_INFO`, and the passenger row count stayed the same. The model can use separate features for each end of a route, such as origin and destination country. Before using those features, we should check whether any of the selected fields are missing.

## 4. External sources

Recommendation only — nothing new is downloaded here. Any source we adopt needs a `DATA.md` entry and Patrick's sign-off first.

- **Census population (metro area):** measures market size at each end of a route, which `pax` and `AIRPORT_INFO` don't capture. Test it first, because larger markets may generate more passengers.
- **FAA enplanements (per airport):** measures total airport traffic, including other carriers. Test it second, because busy hubs can serve more travelers than local population alone suggests.

For both, join only figures published before the forecast date so the features don't leak.

## Findings → modelling consequence (Xiaojie)

This table is what the team's feature set (05) is built from. Every row needs a **consequence**,
not just an observation — "*therefore we should ___ when we model.*"

| # | Observation (with the number) | Modelling consequence |
|---|---|---|
| 1 | Of the 8 columns audited, 4 are candidate features, 1 is the target, and 3 should be dropped from the model inputs. OD_REV_USD contains actual revenue for the forecast month. | Keep departure year and month. Keep origin and destination codes if the route list is known at forecast time. Use OD_PAX as y, and exclude OD_REV_USD, Unnamed: 0, and SF_LOAD_TMS from X. |
| 2 | There are 150 distinct origin codes, 150 distinct destination codes, and 17119 distinct directed routes. One-hot encoding routes would create about 57 times as many columns as encoding the two airport columns separately. | Start by one-hot encoding origin and destination separately. This keeps the feature set smaller and lets the model learn from airports that appear across multiple routes. |
| 3 | The lookup had 0 duplicate nonmissing airport codes. Joining on AIRPRT_CD gave 0 unmatched origin rows and 0 unmatched destination rows, with 0 change in the passenger row count. | Left-join the airport lookup twice in `05`, using separate origin and destination prefixes. This adds airport attributes while preserving one row per original passenger record. Check the selected attributes for missing values before adding them to `X`. |
| 4 | Of the 4 sources reviewed, Census population measures metro area size, and FAA enplanements measure airport traffic. | Test population first because larger markets may generate more passengers. Then test enplanements because busy hubs can serve more travelers than local population alone would suggest. Use only figures published before the forecast date.|